!pip install fastapi uvicorn langchain langchain-openai openai qdrant-client pydub python-dotenv

In [ ]:
import os
from typing import List
import uvicorn
from fastapi import FastAPI
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
from pydub import AudioSegment
from dotenv import load_dotenv

# LangChain & Qdrant Imports
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.tools import tool
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

load_dotenv()


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Add OPENAI_API_KEY=your_key to the .env file."
    )

os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY loaded from .env")

In [ ]:
# ---------------------------------------------------------------------------
# 1. SETUP & DIRECTORIES
# ---------------------------------------------------------------------------
OS_CLIPS_DIR = "static/clips"
OS_AUDIO_DIR = "storage/audio"
os.makedirs(OS_CLIPS_DIR, exist_ok=True)
os.makedirs(OS_AUDIO_DIR, exist_ok=True)

app = FastAPI(title="Audio Transcript Agent")
app.mount("/static", StaticFiles(directory="static"), name="static")

In [ ]:
# ---------------------------------------------------------------------------
# 2. QDRANT COLLECTION
# ---------------------------------------------------------------------------
qdrant_client = QdrantClient(":memory:")
embeddings = OpenAIEmbeddings()

COLLECTION_NAME = "patient_transcripts"

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

In [ ]:
# ---------------------------------------------------------------------------
# 3. DEFINE TOOLS
# ---------------------------------------------------------------------------
@tool
def search_transcript_segments(topic_query: str) -> str:
    """Search real timestamped transcript segments by semantic similarity."""
    query_vector = embeddings.embed_query(topic_query)
    search_results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=5,
    ).points

    if not search_results:
        return "No matching transcript segments found."

    results = []
    for hit in search_results:
        results.append({
            "text": hit.payload["text"],
            "start_time": hit.payload["start_time"],
            "end_time": hit.payload["end_time"],
            "file_path": hit.payload["file_path"],
        })
    return json.dumps(results)


@tool
def extract_audio_clip(file_path: str, start_time: float, end_time: float) -> str:
    """Cut an audio file between start_time and end_time and return its URL."""
    start_ms = int(start_time * 1000)
    end_ms = int(end_time * 1000)
    output_filename = f"clip_{Path(file_path).stem}_{start_time:.1f}_{end_time:.1f}.wav"
    output_path = os.path.join(OS_CLIPS_DIR, output_filename)

    audio = AudioSegment.from_file(file_path)
    clipped = audio[start_ms:end_ms]
    clipped.export(output_path, format="wav")
    return f"/static/clips/{output_filename}"

In [ ]:
# ---------------------------------------------------------------------------
# 3. LOAD TRANSCRIPTS FROM storage/json AND INDEX REAL SEGMENTS
# ---------------------------------------------------------------------------
import json
from pathlib import Path

AUDIO_DIR = Path("storage/audio")
JSON_DIR = Path("storage/json")
LEGACY_TRANSCRIPTS_PATH = Path("storage/transcripts.json")
AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".mp4", ".mpeg", ".mpga", ".webm"}


def load_json_transcripts() -> list[dict]:
    """Load every transcript JSON from storage/json and index the segments for the agent."""
    JSON_DIR.mkdir(parents=True, exist_ok=True)
    transcript_files = sorted(JSON_DIR.glob("*.json"))

    if not transcript_files:
        print(f"No transcript JSON files found in {JSON_DIR}")
        return []

    transcripts = []
    points = []
    point_id = 1

    for transcript_path in transcript_files:
        try:
            payload = json.loads(transcript_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            print(f"Skipping invalid JSON file {transcript_path}: {exc}")
            continue

        if not isinstance(payload, dict):
            print(f"Skipping non-object JSON file {transcript_path}")
            continue

        file_path = payload.get("file_path") or str(AUDIO_DIR / transcript_path.stem)
        segments = payload.get("segments") or []
        transcript_record = {
            "file_path": file_path,
            "text": payload.get("text", ""),
            "segments": [],
        }

        for segment in segments:
            if not isinstance(segment, dict):
                continue
            start_time = float(segment.get("start", 0.0) or 0.0)
            end_time = float(segment.get("end", 0.0) or 0.0)
            text = str(segment.get("text", "")).strip()
            if not text:
                continue

            segment_record = {
                "file_path": file_path,
                "start_time": start_time,
                "end_time": end_time,
                "text": text,
            }
            transcript_record["segments"].append(segment_record)
            points.append(
                PointStruct(
                    id=point_id,
                    vector=embeddings.embed_query(text),
                    payload=segment_record,
                )
            )
            point_id += 1

        transcripts.append(transcript_record)
        print(f"Loaded {transcript_path.name}")

    if points:
        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)

    print(f"Indexed {len(points)} segment(s) from {len(transcripts)} JSON transcript(s)")
    return transcripts


transcripts = load_json_transcripts()
transcripts

In [ ]:
# Initialize Agent
llm = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [search_transcript_segments, extract_audio_clip]

agent_executor = create_agent(
    llm,
    tools,
    system_prompt="""You are an audio intelligence agent.
When a user asks about the audio files:
1. Search the real transcript segments with `search_transcript_segments`.
2. Use the returned file_path, start_time, and end_time with `extract_audio_clip`.
3. Respond with the best top 3matching transcript text, exact timestamps, and an HTML5 audio tag:
   <audio controls src=\"CLIP_URL\"></audio>""",
)

In [ ]:
# ---------------------------------------------------------------------------
# 4. API ENDPOINTS & CHAT INTERFACE
# ---------------------------------------------------------------------------
class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    response: str


@app.post("/api/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    try:
        result = await agent_executor.ainvoke({
            "messages": [{"role": "user", "content": request.message}]
        })
        final_message = result["messages"][-1]
        response_text = final_message.content
        if not isinstance(response_text, str):
            response_text = str(response_text)
        return ChatResponse(response=response_text)
    except Exception as error:
        print(f"Chat request failed: {error}")
        return ChatResponse(response=f"Chat request failed: {error}")


@app.get("/", response_class=HTMLResponse)
async def get_chat_ui():
    return """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Audio Intelligence Agent</title>
        <style>
            body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; max-width: 800px; margin: 40px auto; padding: 0 20px; background: #f7f9fb; }
            .chat-box { border: 1px solid #e1e4e8; border-radius: 8px; background: #fff; height: 500px; overflow-y: auto; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
            .msg { margin-bottom: 20px; padding: 12px 16px; border-radius: 8px; line-height: 1.5; }
            .user { background: #007bff; color: white; margin-left: 20%; }
            .agent { background: #f1f3f5; color: #212529; margin-right: 20%; border: 1px solid #dee2e6; }
            .input-area { display: flex; gap: 10px; margin-top: 15px; }
            input[type="text"] { flex: 1; padding: 12px; border: 1px solid #ced4da; border-radius: 6px; font-size: 14px; }
            button { padding: 12px 24px; background: #007bff; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: 600; }
            button:hover { background: #0056b3; }
            audio { display: block; margin-top: 10px; width: 100%; max-width: 400px; }
        </style>
    </head>
    <body>
        <h2>Audio Intelligence Agent</h2>
        <div class="chat-box" id="chatBox">
            <div class="msg agent">Hello! Ask me to search the transcripts from your audio files.</div>
        </div>
        <div class="input-area">
            <input type="text" id="userInput" placeholder="Ask about the audio transcripts" onkeydown="if(event.key==='Enter') sendMessage()">
            <button onclick="sendMessage()">Send</button>
        </div>

        <script>
            async function sendMessage() {
                const input = document.getElementById('userInput');
                const chatBox = document.getElementById('chatBox');
                const text = input.value.trim();
                if (!text) return;

                chatBox.innerHTML += `<div class="msg user">${escapeHtml(text)}</div>`;
                input.value = '';
                chatBox.scrollTop = chatBox.scrollHeight;

                try {
                    const response = await fetch('/api/chat', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ message: text })
                    });
                    const data = await response.json();
                    if (!response.ok) {
                        throw new Error(data.detail || data.response || `HTTP ${response.status}`);
                    }
                    chatBox.innerHTML += `<div class="msg agent">${data.response}</div>`;
                    chatBox.scrollTop = chatBox.scrollHeight;
                } catch (err) {
                    chatBox.innerHTML += `<div class="msg agent" style="color:red;">Error: ${escapeHtml(err.message)}</div>`;
                }
            }

            function escapeHtml(str) {
                return str.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;");
            }
        </script>
    </body>
    </html>
    """


config = uvicorn.Config(app, host="0.0.0.0", port=8000, reload=False)
server = uvicorn.Server(config)
await server.serve()